In [2]:
pip install OpenAI

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   --------- ---------------------

In [3]:
from openai import OpenAI
from openai import AzureOpenAI
from pathlib import Path
from IPython.display import Audio, display
import json

### gpt4.0 mini client

In [4]:
client = AzureOpenAI(
    api_key="",
    api_version="2024-02-15-preview",
    azure_endpoint=""
)

In [6]:
from IPython.display import Audio

Audio("complain.ogg")

### Speech to text

In [29]:

with open("complain.ogg", "rb") as f:
    response = requests.post(
        "https://team--moixi61d-eastus2.cognitiveservices.azure.com/openai/deployments/whisper/audio/translations?api-version=2024-06-01",
        headers={
            "api-key": "
        },
        files={
            "file": ("complain.ogg", f, "audio/ogg")
        },
        data={
            "model": "whisper",  # 👈 MUST match deployment name
            "prompt": "Nigerian telecom complaint in Pidgin. Preserve meaning exactly."
        }
    )

result = response.json()
print(result)

{'text': "Hello. Good day. I want to complain about my network. For some days now, my data is not working well and calls are dropping. I buy data but I don't use it. This thing doesn't affect my work. I beg you to check it and fix it fast. Thank you."}


### Text to Speech in Pigin plus Summary

In [84]:
import json

system_prompt = """
You are a telecom customer experience assistant in Nigeria.

Tasks:
1. Convert the complaint into natural Nigerian Pidgin
2. Classify the issue into ONE category:
   - network
   - data
   - billing
   - call
   - sim
   - other
3. Detect sentiment:
   - positive
   - neutral
   - frustrated
   - angry
4. Respond in Nigerian Pidgin:
    - Apologize
    - Show understanding
    - Give solution (network troubleshooting or escalation)
    - Keep it short and professional
IMPORTANT:
- Fix any remaining transcription errors
- Do NOT change meaning
- Preserve seriousness

Return STRICT JSON:
{{
  "summary": "...",
  "category": "...",
  "sentiment": "...",
  "response": "..."
}}

Text:
{corrected_text}
"""

response = gpt_client.chat.completions.create(
    model="gpt-4o-mini",  
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": transcription.text}
    ],
    temperature=0
)

raw_output = response.choices[0].message.content

print("RAW OUTPUT:\n", raw_output)

RAW OUTPUT:
 {
  "summary": "I wan complain say my network no dey work well. My data no dey work and calls dey drop. I buy data but I no fit use am.",
  "category": "network",
  "sentiment": "frustrated",
  "response": "I dey sorry for wetin you dey face. I understand say e fit affect your work. Make I check the network issue for you and see wetin I fit do to fix am fast."
}


### Text to speech auto detect but doesn't seem to work well

In [81]:
reply_prompt = f"""
You are a Nigerian telecom customer support agent.

Customer complaint:
{transcription.text}

Instructions:
- Detect the language/style of the complaint
- If it's Pidgin, respond in Pidgin
- If it's English, respond in simple English
- Apologize politely
- Provide a helpful solution
- Keep response short and professional
"""

In [82]:
response = gpt_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": reply_prompt}
    ],
    temperature=0.5
)

spoken_text = response.choices[0].message.content

print(spoken_text)

Hello! I’m really sorry to hear about the issues you’re facing with your network. 

To help fix your data and call dropping problems, please try the following steps:
1. Restart your phone.
2. Check for any software updates and install them.
3. Ensure your network settings are correct.

If the problem persists, please send us your phone number and location, so we can investigate further. Thank you for your patience!


### Just response

In [80]:
import json

system_prompt = """
You are a telecom customer experience assistant in Nigeria.

Tasks:
Respond in Nigerian Pidgin:
- Apologize
- Show understanding
- Give solution (network troubleshooting or escalation)
- Keep it short and professional

Return STRICT JSON:
{{
  "pidgin": "..."
}}

Text:
{corrected_text}
"""

response = gpt_client.chat.completions.create(
    model="gpt-4o-mini",  
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": transcription.text}
    ],
    temperature=0
)

raw_output = response.choices[0].message.content

print("RAW OUTPUT:\n", raw_output)

RAW OUTPUT:
 {
  "pidgin": "I dey sorry for the wahala wey you dey face. I understand say your data no dey work well and calls dey drop. Make I suggest make you try restart your phone first. If e no work, I go escalate am to our technical team to check am well. We go fix am sharp sharp. Thank you for your patience."
}


In [69]:
raw_output = response.choices[0].message.content

import json
analysis_result = json.loads(raw_output)

print(analysis_result)

{'pidgin': 'I dey sorry for the wahala wey you dey face. I understand say your data no dey work well and calls dey drop. Make I suggest say you try restart your phone first. If e no work, I go fit escalate am for you make dem check am well. No worry, we go fix am fast.', 'category': 'network issue', 'sentiment': 'apologetic'}


In [70]:
pidgin_text = analysis_result["pidgin"]

In [72]:
!pip install azure-cognitiveservices-speech

### Speech to text

In [73]:
import azure.cognitiveservices.speech as speechsdk

speech_key = ""
service_region = "eastus2"

speech_config = speechsdk.SpeechConfig(
    subscription=speech_key,
    region=service_region
)

# Optional: choose voice
speech_config.speech_synthesis_voice_name = "en-NG-EzinneNeural"  # Nigerian voice 🇳🇬

audio_config = speechsdk.audio.AudioOutputConfig(filename="response.mp3")

speech_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config,
    audio_config=audio_config
)

# 🔥 Use your pidgin text here
result = speech_synthesizer.speak_text_async(pidgin_text).get()

print("Audio saved as response.mp3")

Audio saved as response.mp3


In [74]:
from IPython.display import Audio

Audio("response.mp3")

### Text to text

In [86]:
system_prompt = """
You are a Nigerian telecom customer support assistant.

Return output in JSON:
{
  "summary": "...",
  "category": "...",
  "sentiment": "...",
  "response": "..."
}

Instructions:
- Detect language (Pidgin or English)
- Respond in same language/style
- Be polite and helpful
- Keep it short
"""

In [89]:
user_input = "Hello, good day. I dey call to complain about my network service. For some days now, my line no dey work well at all.Data dey slow, sometimes e no even connect. I fit buy data, but I no go fit use am. Calls sef dey drop or no go through.This thing don affect my work and communication. I don try restart my phone and even remove SIM, but nothing change.Abeg, make una check wetin dey happen for my area and fix am as soon as possible. If na network issue, una suppose notify customers.I go appreciate quick response because this situation no dey acceptable.Thank you."
    
response = gpt_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_input}
    ],
    temperature=0
)

In [90]:
import json

result = json.loads(response.choices[0].message.content)

print("SUMMARY:", result["summary"])
print("CATEGORY:", result["category"])
print("SENTIMENT:", result["sentiment"])
print("RESPONSE:", result["response"])

SUMMARY: Customer is experiencing poor network service, including slow data and dropped calls, affecting work and communication.
CATEGORY: Network Issue
SENTIMENT: Frustrated
RESPONSE: Hello, thank you for reaching out. We dey sorry for the inconvenience wey you dey face with your network service. We go check the issue for your area and update you as soon as possible. Your feedback dey important to us, and we go work to resolve am quickly. Thank you for your patience.
